# Electric vs conventional bikes

Uses the officially downloaded June 2024 dataset. No cycling paths are inferred here.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
PROJECT = Path.cwd()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
DATA = PROJECT / 'data/processed/bikeshare_2024_06.parquet'
assert DATA.exists(), f'Missing {DATA}; run the download and prepare scripts from the repository root.'
df = pd.read_parquet(DATA)
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
print(f'Loaded {len(df):,} records')

In [ ]:
print('Raw bike model labels:')
print(df['Bike_Model'].value_counts(dropna=False))
def category(value):
    if pd.isna(value): return 'Unknown'
    name=str(value).strip().upper()
    if name.startswith('EFIT'): return 'Electric'
    if name=='ICONIC': return 'Conventional'
    return 'Unknown'
df['bike_type']=df['Bike_Model'].map(category)
df['date']=df['Start_Time'].dt.date
print(df['bike_type'].value_counts())

In [ ]:
daily=df.groupby(['date','bike_type']).size().unstack(fill_value=0)
fig,ax=plt.subplots(figsize=(13,5),facecolor='#080b12')
ax.set_facecolor('#080b12')
for label,color in [('Electric','#00e5ff'),('Conventional','#ff4fd8')]:
    if label in daily:
        ax.plot(pd.to_datetime(daily.index),daily[label],color=color,label=label,lw=2)
ax.set(title='June 2024 | Daily trips by bike type',ylabel='Recorded trips')
ax.title.set_color('white');ax.yaxis.label.set_color('white');ax.tick_params(colors='white');ax.legend()
fig.autofmt_xdate();plt.show()

In [ ]:
counts=daily.reindex(columns=['Electric','Conventional'],fill_value=0)
share=100*counts['Electric']/counts.sum(axis=1).replace(0,np.nan)
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(pd.to_datetime(share.index),share.values,color='#008fb0',marker='o')
ax.set(title='Daily electric-bike share (known types only)',ylabel='Percent',ylim=(0,100));ax.grid(alpha=.2);fig.autofmt_xdate();plt.show()